# FlowGraphFormer
## 00A_Label_Converter.ipynb

Run this **before** `00B_Dataset_Harmonizer.ipynb`, whenever you add a dataset
release that encodes `Label` as an **integer** instead of an attack-name
string -- e.g. CICIoT2023-style exports (`Label` = 0/1, actual attack type
only present in the *filename*, like `DDoS_UDP_Flood.csv`, with the
`attack_name` column empty/NaN for every row).

**Why this is a separate step from the Harmonizer:**

`00B_Dataset_Harmonizer.ipynb`'s label taxonomy (`LABEL_RULES`) matches
*keywords inside the existing `Label` column's text*. If `Label` is just the
integer `1` for every row, there is no text for it to match against -- the
Harmonizer would pass every attack row through as literal `1`, and your
whole taxonomy step (and therefore Cell 8/Feature Selection + SMOTE class
balancing downstream) silently breaks.

**What this notebook does:**

1. Backs up every original raw CSV to `dataset/raw_backup_prelabel/` (never
   destructive -- separate backup folder from the Harmonizer's, so both
   backups survive).
2. Scans every CSV in `dataset/raw/` and **flags** files whose `Label`
   column is numeric (int/float) rather than text -- this is the CICIoT2023
   signature.
3. For flagged files only: derives the real attack name from the **file
   name** (not the data), using the same keyword taxonomy style as the
   Harmonizer (`BENIGN`, `DDoS`, `DoS`, `PortScan`, `Bot`, `Infiltration`,
   `BruteForce`, `WebAttack`, `Spoofing`, ...).
4. Rewrites `Label`: `0` -> `BENIGN`, any non-zero value -> the
   filename-derived attack string. Drops the (empty) `attack_name` column
   if present, since it gets folded into `Label`.
5. Leaves files that already have a text `Label` column (CICIDS2017/2018
   style) **completely untouched**.
6. Saves a report to `reports/label_conversion/` so you can see
   exactly which files were converted, what label they were assigned, and
   which need manual review (e.g. files with more than one non-zero integer
   value, where a single filename-derived label may not be precise enough).

**After this runs, run `00B_Dataset_Harmonizer.ipynb` next, then
`01_Preprocessing_local.ipynb` as usual.**


In [ ]:
# ==============================================================================
# FlowGraphFormer
# 00A_Label_Converter.ipynb
#
# Cell 0 : Environment Detection & Initialization (Colab + Local)
# ==============================================================================

import os
import sys
import gc
import re
import json
import shutil
import warnings
import subprocess
import importlib
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

print("=" * 80)
print("FlowGraphFormer")
print("Research Notebook : 00A_Label_Converter")
print("=" * 80)

# ------------------------------------------------------------------------------
# Detect Runtime (Colab vs Local)
# ------------------------------------------------------------------------------

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

print(f"Runtime Detected      : {'Google Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

PROJECT_NAME = "FlowGraphFormer"

if IN_COLAB:
    PROJECT_ROOT = Path("/content/drive/MyDrive/ML_Project") / PROJECT_NAME
else:
    _cwd = Path.cwd().resolve()
    if _cwd.name == PROJECT_NAME:
        PROJECT_ROOT = _cwd
    elif (_cwd / PROJECT_NAME).exists():
        PROJECT_ROOT = _cwd / PROJECT_NAME
    else:
        PROJECT_ROOT = _cwd

print(f"Project Root          : {PROJECT_ROOT}")

# ------------------------------------------------------------------------------
# Install Missing Packages
# ------------------------------------------------------------------------------

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "tqdm": "tqdm",
}

missing_packages = []
for module, package in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(module)
    except ImportError:
        missing_packages.append(package)

if missing_packages:
    print(f"\nInstalling missing packages: {missing_packages}\n")
    pip_cmd = [sys.executable, "-m", "pip", "install", "-q"]
    if not IN_COLAB:
        pip_cmd.append("--break-system-packages")
    subprocess.check_call(pip_cmd + missing_packages)
else:
    print("\nAll required packages already installed.\n")

import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# ------------------------------------------------------------------------------
# Directories (same layout as 00B_Dataset_Harmonizer.ipynb / 01_Preprocessing_local.ipynb)
# ------------------------------------------------------------------------------

DATASET_DIR = PROJECT_ROOT / "dataset"
RAW_DATASET_DIR = DATASET_DIR / "raw"
RAW_BACKUP_DIR = DATASET_DIR / "raw_backup_prelabel"
REPORT_DIR = PROJECT_ROOT / "reports" / "label_conversion"

for directory in [RAW_DATASET_DIR, RAW_BACKUP_DIR, REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Raw Dataset Directory  : {RAW_DATASET_DIR}")
print(f"Backup Directory       : {RAW_BACKUP_DIR}")
print(f"Report Directory       : {REPORT_DIR}")
print("\nNotebook Ready.")
print("=" * 80)


In [ ]:
# ==============================================================================
# Cell 1 : Configuration
# ==============================================================================

CONFIG = {

    # Name of the label column to inspect/convert. Case-insensitive match
    # is used when scanning, so "label", "Label", "LABEL" all work.
    "label_column_candidates": ["label", "Label", "LABEL"],

    # Column that CICIoT2023-style exports sometimes ship alongside the
    # integer Label (usually all-NaN in single-attack-per-file exports).
    # Dropped after conversion since its information is folded into Label.
    "redundant_columns": ["attack_name", "Attack_name", "Attack_Name", "attack_type"],

    # A file's Label column is treated as "needs conversion" if it is
    # numeric (int/float dtype) rather than text/object dtype.
    # This is the CICIoT2023 signature vs. the CICIDS2017/2018 signature.
    "numeric_label_triggers_conversion": True,

    # Benign rows are conventionally encoded as 0 in CICIoT2023-style
    # files. If a file uses a different convention, add an override here
    # keyed by filename (exact match).
    "benign_value_default": 0,
    "benign_value_overrides": {
        # "some_weird_file.csv": 1,
    },

    # Overwrite files in dataset/raw/ in place (recommended -- keeps
    # 00B_Dataset_Harmonizer.ipynb / 01_Preprocessing_local.ipynb unchanged).
    # Originals are always backed up first regardless of this setting.
    "overwrite_in_place": True,

    # If True, no files are written -- only the report is produced, so you
    # can sanity-check the derived labels before committing to them.
    "dry_run": False,

}

print("Configuration loaded.")
print(json.dumps({k: v for k, v in CONFIG.items()}, indent=2))


In [ ]:
# ==============================================================================
# Cell 2 : Filename-Based Label Taxonomy
# ==============================================================================
# CICIoT2023 (and similar integer-labeled releases) ship one attack type
# per file, with the true attack name only present in the FILENAME
# (e.g. "DDoS_UDP_Flood.csv", "DoS-SYN_Flood.csv", "Recon-PortScan.csv").
# The `Label`/`attack_name` columns themselves are 0/1 (or all-NaN for
# attack_name) and carry no usable text.
#
# Keyword groups follow the same style/order-matters-first-match-wins
# pattern as 00B_Dataset_Harmonizer.ipynb's LABEL_RULES, extended with
# CICIoT2023-specific attack families (Mirai/botnet, spoofing, recon,
# flood variants) so the OUTPUT vocabulary lines up with CIC-IDS2017/2018
# labels wherever the attack concept genuinely overlaps.

FILENAME_LABEL_RULES = [

    (["benign", "normal"], "BENIGN"),

    (["heartbleed"], "Heartbleed"),

    (["udp flood", "udpflood", "syn flood", "synflood",
      "ddos", "distributed denial", "flood"], "DDoS"),

    (["dos slowloris", "slowloris"], "DoS"),
    (["dos slowhttptest", "slowhttptest"], "DoS"),
    (["dos hulk", "hulk"], "DoS"),
    (["dos goldeneye", "goldeneye"], "DoS"),
    (["dos "], "DoS"),

    (["portscan", "port scan", "recon", "reconnaissance",
      "vulnerabilityscan", "hostdiscovery", "osscan", "pingsweep"], "PortScan"),

    (["mirai", "botnet", "bot "], "Bot"),

    (["infiltration", "backdoor"], "Infiltration"),

    (["patator", "brute force", "bruteforce", "brute-force",
      "dictionary", "credential"], "BruteForce"),

    (["sql injection", "sqli"], "WebAttack"),
    (["xss", "cross site"], "WebAttack"),
    (["web attack", "webattack", "commandinjection", "uploading"], "WebAttack"),

    (["spoof", "mitm", "arp", "dns spoof"], "Spoofing"),

    (["malware", "ransomware"], "Malware"),
]


def normalize_filename(fname):
    """Strip extension and separators, lowercase -- ready for keyword match."""
    name = re.sub(r"\.(csv|parquet)$", "", str(fname), flags=re.IGNORECASE)
    name = re.sub(r"[_\-]+", " ", name)
    name = re.sub(r"\s+", " ", name)
    return name.strip().lower()


def derive_label_from_filename(fname):
    """Returns (label, method) where method is 'mapped' or 'unmapped'.
    On 'unmapped', label falls back to a cleaned version of the filename
    itself, and the file is flagged in the report for manual review."""
    cleaned = normalize_filename(fname)
    for keywords, canonical in FILENAME_LABEL_RULES:
        if any(kw in cleaned for kw in keywords):
            return canonical, "mapped"
    # Fallback: title-cased filename stem, no spaces -- still usable as a
    # class label, just not guaranteed to match CIC-IDS2017 vocabulary.
    fallback = cleaned.title().replace(" ", "")
    return fallback, "unmapped"


print("Filename label taxonomy  :", len(FILENAME_LABEL_RULES), "keyword groups")


In [ ]:
# ==============================================================================
# Cell 3 : Scan dataset/raw/ and Flag Files Needing Conversion
# ==============================================================================

print("=" * 80)
print("Scanning dataset/raw/ for CSV files")
print("=" * 80)

csv_files = sorted(RAW_DATASET_DIR.glob("*.csv"))

if len(csv_files) == 0:
    raise FileNotFoundError(f"No CSV files found in:\n{RAW_DATASET_DIR}")

print(f"Found {len(csv_files)} CSV file(s).\n")


def find_label_column(columns):
    lookup = {c.strip().lower(): c for c in columns}
    for candidate in CONFIG["label_column_candidates"]:
        if candidate.strip().lower() in lookup:
            return lookup[candidate.strip().lower()]
    return None


scan_report = []

for file in tqdm(csv_files, desc="Scanning Files"):

    try:
        # Only need the header + Label column dtype, not the whole file,
        # to decide whether conversion is needed -- much faster/lighter
        # than a full read for large CICIoT2023 exports.
        header_df = pd.read_csv(file, nrows=5000, low_memory=False)
    except Exception as e:
        scan_report.append({
            "File": file.name, "Label_Column": None,
            "Label_Dtype": None, "Needs_Conversion": False,
            "Status": f"Failed to read ({e})",
        })
        continue

    label_col = find_label_column(header_df.columns)

    if label_col is None:
        scan_report.append({
            "File": file.name, "Label_Column": None,
            "Label_Dtype": None, "Needs_Conversion": False,
            "Status": "No Label column found -- SKIPPED",
        })
        continue

    is_numeric = pd.api.types.is_numeric_dtype(header_df[label_col])
    needs_conversion = is_numeric and CONFIG["numeric_label_triggers_conversion"]

    derived_label, method = derive_label_from_filename(file.name)

    scan_report.append({
        "File": file.name,
        "Label_Column": label_col,
        "Label_Dtype": str(header_df[label_col].dtype),
        "Sample_Values": sorted(header_df[label_col].dropna().unique().tolist())[:5],
        "Needs_Conversion": needs_conversion,
        "Derived_Label_From_Filename": derived_label if needs_conversion else None,
        "Filename_Match_Method": method if needs_conversion else None,
        "Status": "Flagged for conversion" if needs_conversion else "Already text-labeled -- untouched",
    })

scan_df = pd.DataFrame(scan_report)
display(scan_df)

n_flagged = int(scan_df["Needs_Conversion"].sum()) if len(scan_df) else 0
n_unmapped = int((scan_df.get("Filename_Match_Method") == "unmapped").sum()) if len(scan_df) else 0

print(f"\nFiles flagged for conversion : {n_flagged} / {len(scan_df)}")
if n_unmapped > 0:
    print(f"WARNING: {n_unmapped} file(s) had no filename keyword match -- "
          f"review 'Derived_Label_From_Filename' for those rows before proceeding.")

scan_df.to_csv(REPORT_DIR / "label_scan_report.csv", index=False)


In [ ]:
# ==============================================================================
# Cell 4 : Convert Flagged Files (Backup -> Rewrite Label -> Save)
# ==============================================================================

print("=" * 80)
print("Converting Integer Labels to CIC-IDS-Style Text Labels")
print("=" * 80)

files_to_convert = scan_df.loc[scan_df["Needs_Conversion"], "File"].tolist()

if len(files_to_convert) == 0:
    print("\nNo files needed conversion -- nothing to do.")
else:
    conversion_report = []

    for fname in tqdm(files_to_convert, desc="Converting Files"):
        file_path = RAW_DATASET_DIR / fname

        try:
            df = pd.read_csv(file_path, low_memory=False)
            label_col = find_label_column(df.columns)

            # ------------------------------------------------------------
            # Backup original BEFORE any modification
            # ------------------------------------------------------------
            backup_path = RAW_BACKUP_DIR / fname
            if not backup_path.exists():
                shutil.copy2(file_path, backup_path)

            # ------------------------------------------------------------
            # Derive attack label from filename (not from the data)
            # ------------------------------------------------------------
            attack_label, method = derive_label_from_filename(fname)
            benign_value = CONFIG["benign_value_overrides"].get(
                fname, CONFIG["benign_value_default"]
            )

            distinct_values = df[label_col].dropna().unique()
            n_nonbenign_values = len([v for v in distinct_values if v != benign_value])

            # ------------------------------------------------------------
            # Rewrite Label: benign_value -> BENIGN, everything else ->
            # the single filename-derived attack label.
            # ------------------------------------------------------------
            df[label_col] = np.where(
                df[label_col] == benign_value, "BENIGN", attack_label
            )
            df.rename(columns={label_col: "Label"}, inplace=True)

            # ------------------------------------------------------------
            # Drop redundant/empty attack-name columns
            # ------------------------------------------------------------
            drop_cols = [c for c in CONFIG["redundant_columns"] if c in df.columns]
            if drop_cols:
                df.drop(columns=drop_cols, inplace=True)

            label_counts = df["Label"].value_counts().to_dict()

            if not CONFIG["dry_run"]:
                df.to_csv(file_path, index=False)

            conversion_report.append({
                "File": fname,
                "Derived_Attack_Label": attack_label,
                "Match_Method": method,
                "Multiple_NonBenign_Values_In_Source": n_nonbenign_values > 1,
                "Label_Distribution": label_counts,
                "Status": "Converted" if not CONFIG["dry_run"] else "Dry-run (not written)",
            })

            del df
            gc.collect()

        except Exception as e:
            conversion_report.append({
                "File": fname, "Derived_Attack_Label": None,
                "Match_Method": None,
                "Multiple_NonBenign_Values_In_Source": None,
                "Label_Distribution": None,
                "Status": f"FAILED ({e})",
            })

    conversion_df = pd.DataFrame(conversion_report)
    display(conversion_df)

    n_multi = int(conversion_df["Multiple_NonBenign_Values_In_Source"].fillna(False).sum())
    if n_multi > 0:
        print(f"\nWARNING: {n_multi} file(s) had MORE THAN ONE non-benign integer value "
              f"in the source data, but were collapsed to a single filename-derived label. "
              f"Open those files' 'Label_Distribution' and confirm this is correct for your "
              f"dataset release before running the Harmonizer.")

    report_path = REPORT_DIR / "label_conversion_report.csv"
    conversion_df.to_csv(report_path, index=False)
    print(f"\nConversion report saved to:\n{report_path}")


In [ ]:
# ==============================================================================
# Cell 5 : Post-Conversion Verification
# ==============================================================================

print("=" * 80)
print("Post-Conversion Verification")
print("=" * 80)

verify_report = []

for file in tqdm(csv_files, desc="Verifying Files"):
    try:
        head_df = pd.read_csv(file, nrows=5000, low_memory=False)
        label_col = find_label_column(head_df.columns)
        is_text = label_col is not None and not pd.api.types.is_numeric_dtype(head_df[label_col])
        verify_report.append({
            "File": file.name,
            "Label_Column": label_col,
            "Is_Text_Label_Now": bool(is_text),
            "Sample_Labels": sorted(head_df[label_col].dropna().unique().tolist())[:6] if label_col else [],
        })
    except Exception as e:
        verify_report.append({
            "File": file.name, "Label_Column": None,
            "Is_Text_Label_Now": False, "Sample_Labels": f"Failed ({e})",
        })

verify_df = pd.DataFrame(verify_report)
display(verify_df)

all_text = bool(verify_df["Is_Text_Label_Now"].all())
print("\n" + "=" * 80)
if all_text:
    print("PASS -- every file in dataset/raw/ now has a text-based Label column.")
else:
    failing = verify_df.loc[~verify_df["Is_Text_Label_Now"], "File"].tolist()
    print("REVIEW NEEDED -- the following file(s) still have a non-text Label column:")
    for f in failing:
        print(f"  - {f}")
print("=" * 80)

verify_df.to_csv(REPORT_DIR / "post_conversion_verification.csv", index=False)

print(f"\nOriginal (pre-conversion) files backed up to:\n{RAW_BACKUP_DIR}")
print("\nNext step: run 00B_Dataset_Harmonizer.ipynb, then 01_Preprocessing_local.ipynb.")
print("=" * 80)
